# 🌊 Transmembrane Protein Analysis

---

## Learning Objectives

- Understand membrane protein topology
- Predict transmembrane helices from sequence
- Use hydropathy analysis
- Interpret topology prediction results

---

## Membrane Protein Architecture

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    MEMBRANE PROTEIN TOPOLOGY                            │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Extracellular                                                         │
│   ════════════════════════════════════════════════════════════════════  │
│        N-term          Loop1            Loop2                           │
│          │               │                │                             │
│          ↓               ↓                ↓                             │
│   ~~~~~~~~~~~────────│─────│────────│─────│───────~~~~~~~~~~~~~~~~~~~~  │
│   Lipid ▓▓▓▓▓▓▓▓▓▓▓▓│░░░░░│▓▓▓▓▓▓▓▓│░░░░░│▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓ Bilayer     │
│   bilayer▓▓▓▓▓▓▓▓▓▓▓│░TM1░│▓▓▓▓▓▓▓▓│░TM2░│▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓             │
│         ▓▓▓▓▓▓▓▓▓▓▓▓│░░░░░│▓▓▓▓▓▓▓▓│░░░░░│▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓             │
│   ~~~~~~~~~~~────────│─────│────────│─────│───────~~~~~~~~~~~~~~~~~~~~  │
│                              ↑                ↑                         │
│                            Loop             C-term                      │
│   Cytoplasm (inside)                                                    │
│   ════════════════════════════════════════════════════════════════════  │
│                                                                         │
│   TM helix characteristics:                                             │
│   • ~20-25 hydrophobic amino acids                                      │
│   • α-helical structure (hydrogen bonds satisfied)                      │
│   • Rich in Leu, Ile, Val, Ala, Phe                                    │
│   • Spans ~30Å membrane thickness                                       │
│                                                                         │
│   Topology types:                                                       │
│   ┌────────────┬─────────────────────────────────────────┐              │
│   │ Type I     │ N-out, C-in, single TM                 │              │
│   │ Type II    │ N-in, C-out, single TM                 │              │
│   │ Multi-pass │ Multiple TM helices (e.g., GPCRs: 7TM) │              │
│   │ β-barrel   │ Outer membrane proteins (bacteria)     │              │
│   └────────────┴─────────────────────────────────────────┘              │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Kyte-Doolittle hydropathy scale
HYDROPATHY = {
    'A': 1.8,  'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8,  'K': -3.9, 'M': 1.9,  'F': 2.8,  'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

def calculate_hydropathy(sequence, window_size=19):
    """
    Calculate hydropathy profile using sliding window.
    
    Window size 19 is typical for TM helix prediction.
    Values >1.6 suggest TM region.
    """
    profile = []
    half_window = window_size // 2
    
    for i in range(half_window, len(sequence) - half_window):
        window = sequence[i - half_window:i + half_window + 1]
        avg_hydropathy = sum(HYDROPATHY.get(aa, 0) for aa in window) / window_size
        profile.append({
            'position': i + 1,  # 1-indexed
            'hydropathy': avg_hydropathy
        })
    
    return profile

def predict_tm_regions(sequence, window_size=19, threshold=1.6, min_length=15):
    """
    Predict transmembrane regions from hydropathy profile.
    """
    profile = calculate_hydropathy(sequence, window_size)
    
    # Find regions above threshold
    in_tm = False
    tm_start = None
    tm_regions = []
    
    for point in profile:
        if point['hydropathy'] > threshold:
            if not in_tm:
                tm_start = point['position']
                in_tm = True
        else:
            if in_tm:
                tm_end = point['position'] - 1
                if tm_end - tm_start >= min_length:
                    tm_regions.append((tm_start, tm_end))
                in_tm = False
    
    # Handle region at end
    if in_tm:
        tm_end = profile[-1]['position']
        if tm_end - tm_start >= min_length:
            tm_regions.append((tm_start, tm_end))
    
    return tm_regions, profile

# Example: Rhodopsin-like GPCR transmembrane segments
# Simplified sequence with hydrophobic TM regions
example_seq = (
    "MNGTEGLEV" +  # N-terminus
    "LLIVFILAG" * 2 + "VILL" +  # TM1
    "ERKQTW" +  # Loop
    "AVVLAFLLW" * 2 + "ILAM" +  # TM2
    "DRSQWYNPK" +  # Loop
    "FFILCALLI" * 2 + "VLFL" +  # TM3
    "QSKTEVNDH"  # C-terminus
)

tm_regions, profile = predict_tm_regions(example_seq)

print("Hydropathy Analysis:")
print("=" * 50)
print(f"Sequence length: {len(example_seq)} aa")
print(f"\nPredicted TM regions:")
for i, (start, end) in enumerate(tm_regions, 1):
    segment = example_seq[start-1:end]
    print(f"  TM{i}: {start}-{end} ({end-start+1} aa)")
    print(f"        {segment}")

In [ ]:
def visualize_hydropathy_ascii(profile, sequence, tm_regions):
    """
    Create ASCII visualization of hydropathy profile.
    """
    # Scale values to fit in 20 character height
    min_val = min(p['hydropathy'] for p in profile)
    max_val = max(p['hydropathy'] for p in profile)
    range_val = max_val - min_val
    
    height = 15
    width = min(len(profile), 60)  # Limit width
    step = max(1, len(profile) // width)
    
    # Sample profile
    sampled = profile[::step][:width]
    
    # Create plot grid
    grid = [[' ' for _ in range(len(sampled))] for _ in range(height)]
    
    # Add threshold line
    threshold = 1.6
    thresh_row = int((max_val - threshold) / range_val * (height - 1))
    for col in range(len(sampled)):
        if 0 <= thresh_row < height:
            grid[thresh_row][col] = '-'
    
    # Plot values
    for col, point in enumerate(sampled):
        row = int((max_val - point['hydropathy']) / range_val * (height - 1))
        if 0 <= row < height:
            grid[row][col] = '█' if point['hydropathy'] > threshold else '▒'
    
    # Print
    print("\nHydropathy Plot:")
    print(f"{max_val:5.1f} │", end='')
    print(''.join(grid[0]))
    
    for i, row in enumerate(grid[1:-1], 1):
        if i == thresh_row:
            print("  1.6 │", end='')
        else:
            print("      │", end='')
        print(''.join(row))
    
    print(f"{min_val:5.1f} │", end='')
    print(''.join(grid[-1]))
    print("      └" + "─" * len(sampled))
    print("        Position →")
    
    print("\nLegend: █ = hydrophobic (TM), ▒ = hydrophilic, --- = threshold")

visualize_hydropathy_ascii(profile, example_seq, tm_regions)

---

## Positive-Inside Rule

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    POSITIVE-INSIDE RULE                                 │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Cytoplasmic loops have more positive charges (Arg, Lys)               │
│   than extracellular loops                                              │
│                                                                         │
│   Extracellular (outside)                                               │
│   ─────────────────────────────────────────────────────────────────     │
│         │                     │                                         │
│         │  Low K+R content    │                                         │
│   ══════╪═════════════════════╪══════════════════════════════════ Mem   │
│         │░░░░░░░░░░░░░░░░░░░░░│                                         │
│         │  High K+R content   │                                         │
│   ─────────────────────────────────────────────────────────────────     │
│   Cytoplasm (inside)         ↑                                          │
│                              │                                          │
│                    Positive charges accumulate                          │
│                    in cytoplasmic loops                                 │
│                                                                         │
│   This rule helps determine protein orientation:                        │
│   • Count K+R in loops between TM segments                              │
│   • Loops with more positive charges face cytoplasm                     │
│   • ~75% accuracy for topology prediction                               │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def analyze_topology(sequence, tm_regions):
    """
    Predict topology using positive-inside rule.
    """
    # Extract loops
    loops = []
    prev_end = 0
    
    for start, end in tm_regions:
        if start > prev_end + 1:
            loop_seq = sequence[prev_end:start-1]
            loops.append({
                'start': prev_end + 1,
                'end': start - 1,
                'sequence': loop_seq,
                'length': len(loop_seq)
            })
        prev_end = end
    
    # C-terminal region
    if prev_end < len(sequence):
        loop_seq = sequence[prev_end:]
        loops.append({
            'start': prev_end + 1,
            'end': len(sequence),
            'sequence': loop_seq,
            'length': len(loop_seq)
        })
    
    # Calculate positive charges in each loop
    for loop in loops:
        positive_count = sum(1 for aa in loop['sequence'] if aa in 'KR')
        loop['positive_charges'] = positive_count
        loop['density'] = positive_count / loop['length'] if loop['length'] > 0 else 0
    
    # Assign inside/outside based on alternating pattern + positive-inside
    # First, determine which side has more positive charges
    odd_loops = [loops[i] for i in range(0, len(loops), 2)]  # 1st, 3rd, etc.
    even_loops = [loops[i] for i in range(1, len(loops), 2)]  # 2nd, 4th, etc.
    
    odd_positive = sum(l['positive_charges'] for l in odd_loops)
    even_positive = sum(l['positive_charges'] for l in even_loops)
    
    # Assign orientations
    if odd_positive >= even_positive:
        # N-terminus is inside
        orientation = 'N-in'
        for i, loop in enumerate(loops):
            loop['location'] = 'inside' if i % 2 == 0 else 'outside'
    else:
        # N-terminus is outside
        orientation = 'N-out'
        for i, loop in enumerate(loops):
            loop['location'] = 'outside' if i % 2 == 0 else 'inside'
    
    return {
        'orientation': orientation,
        'loops': loops,
        'tm_count': len(tm_regions)
    }

topology = analyze_topology(example_seq, tm_regions)

print("Topology Analysis:")
print("=" * 50)
print(f"Orientation: {topology['orientation']}")
print(f"TM helices: {topology['tm_count']}")
print(f"\nLoop analysis:")
for loop in topology['loops']:
    print(f"  {loop['start']}-{loop['end']}: {loop['location']}")
    print(f"    K+R count: {loop['positive_charges']} in {loop['length']} aa")

---

## Signal Peptides and Anchors

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    SIGNAL SEQUENCES                                     │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Signal peptide (secretory/membrane proteins):                         │
│                                                                         │
│   N-terminus                                                            │
│   ├───────────┬───────────────┬──────────┬──────────────────────────    │
│   │  n-region │   h-region    │ c-region │  Mature protein              │
│   │  (+)      │  hydrophobic  │   A-X-A  │                              │
│   ├───────────┴───────────────┴──────────┴──────────────────────────    │
│   │  1-5 aa   │   7-15 aa     │  3-7 aa  │                              │
│   │  basic    │   Leu-rich    │ cleavage │                              │
│                      ↓              ↓                                   │
│               ER targeting    Signal peptidase                          │
│                               cleavage site                             │
│                                                                         │
│   Membrane anchors:                                                     │
│   ┌──────────────────────────────────────────────────────────────┐      │
│   │ Type           │ Description                                 │      │
│   ├────────────────┼─────────────────────────────────────────────│      │
│   │ GPI anchor     │ C-terminal, lipid-attached                  │      │
│   │ Myristoylation │ N-terminal Gly, co-translational            │      │
│   │ Palmitoylation │ Internal Cys, reversible                    │      │
│   │ Prenylation    │ C-terminal CaaX motif                       │      │
│   └──────────────────────────────────────────────────────────────┘      │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def predict_signal_peptide(sequence, max_length=30):
    """
    Simple signal peptide prediction.
    Real prediction requires machine learning (SignalP).
    """
    n_term = sequence[:max_length]
    
    # Check for characteristics
    features = {
        'n_region': {'start': 0, 'end': 5, 'positive': 0},
        'h_region': {'start': 5, 'end': 20, 'hydrophobic': 0},
        'c_region': {'start': 17, 'end': 25, 'small_aa': 0}
    }
    
    # Count positive in n-region
    features['n_region']['positive'] = sum(1 for aa in n_term[:5] if aa in 'RK')
    
    # Count hydrophobic in h-region
    features['h_region']['hydrophobic'] = sum(1 for aa in n_term[5:20] if aa in 'ILVFMAW')
    
    # Check for -3,-1 rule (Ala-X-Ala pattern)
    axa_found = False
    cleavage_site = None
    for i in range(17, min(25, len(n_term)-2)):
        if n_term[i] in 'AS' and n_term[i+2] in 'AS':
            axa_found = True
            cleavage_site = i + 3
            break
    
    # Score
    score = 0
    if features['n_region']['positive'] >= 1:
        score += 2
    if features['h_region']['hydrophobic'] >= 8:
        score += 3
    if axa_found:
        score += 2
    
    return {
        'has_signal_peptide': score >= 5,
        'confidence': score / 7,
        'cleavage_site': cleavage_site,
        'features': features,
        'n_term_sequence': n_term
    }

# Example: Insulin signal peptide
insulin_precursor = "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPK"
result = predict_signal_peptide(insulin_precursor)

print("Signal Peptide Prediction:")
print("=" * 50)
print(f"Sequence: {result['n_term_sequence'][:30]}...")
print(f"Signal peptide: {'Yes' if result['has_signal_peptide'] else 'No'}")
print(f"Confidence: {result['confidence']:.1%}")
if result['cleavage_site']:
    print(f"Cleavage site: position {result['cleavage_site']}")
    print(f"Mature protein starts: {insulin_precursor[result['cleavage_site']:result['cleavage_site']+10]}...")

---

## 🏋️ Exercises

### Exercise 1: Hydropathy Analysis
Calculate hydropathy for a human GPCR sequence.

### Exercise 2: Topology Prediction
Predict topology and verify against UniProt annotation.

### Exercise 3: Compare Methods
Compare simple hydropathy with TMHMM predictions.

---

## 📚 Resources

- [TMHMM](https://services.healthtech.dtu.dk/service.php?TMHMM-2.0) - TM topology prediction
- [SignalP](https://services.healthtech.dtu.dk/service.php?SignalP) - Signal peptide prediction
- [Phobius](https://phobius.sbc.su.se/) - Combined TM and signal prediction
- [OPM](https://opm.phar.umich.edu/) - Membrane protein database

---
**Navigation:** [Back to Course README](../../README.md) · [Open this notebook path](`Course/Archive/07_Kodomo_Structural_Bioinformatics/13_Functional_Annotation/03_Transmembrane_Proteins.ipynb`)
